# 12 -- Segmentación por volumen (fase E, parte 1)

Motivación (09 y análisis de concentración): el volumen está muy concentrado (un producto es ~15% del volumen de prueba;
182 de 6.427 pares mueven la mitad) y el modelo global falla justo en esa cabeza. Preguntas:

1. ¿Conviene **modelar aparte** los pares de mayor volumen?
2. ¿Qué **modelo** sirve mejor en cada estrato (global, relativo, solo del estrato, por serie)?
3. ¿El valor de las **features** cambia por estrato (es decir, la segmentación cambia el feature engineering)?

Diseño para no repetir errores anteriores:
- **Estratos definidos as-of**: por fold, con el volumen de los últimos 365 días hábiles **hasta `train_end`** (nunca con el volumen del periodo
  de prueba). Cabeza = los pares que acumulan el 50% del volumen, medio = el siguiente 30%, cola = el 20% restante.
- Entrenamiento con **purge** (`purgar_entrenamiento`), mismas filas y folds que 08; hiperparámetros fijos (sin búsqueda), IC95 por
  bootstrap de productos, política de holdout de `evaluar_nivel1` (se elige con los folds 2-4 y se confirma en el 5).
- Referencia de comparación para la segmentación: el modelo **global relativo** `rel` (el mejor global de la fase D), además del piso.

Candidatos: `piso`, `v1`, `rel`, `rel_w`, `res_mm`, `mezcla50` (fases anteriores); `mm_cal` y `ewma_cal` (media móvil / EWMA por el factor de
calendario, sin entrenar); `ets` (Holt-Winters aditivo con estacionalidad semanal por serie, solo cabeza y medio); `seg_v1` y `seg_v2`
(modelo relativo entrenado **solo con las filas del estrato**, con las 14 features v1 o con las 25 de v2).

Cadena: ... -> `07b` -> `07c` -> `11` -> **`12_segmentacion_volumen`** -> `13_cuantiles_y_costos`

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import (DW, registrar_huella, verificar_huella, leer_panel, leer_calendario_habil,
                                 purgar_entrenamiento)
import evaluar_nivel1 as ev
from exportar_modelos_nivel1 import FEATURES as F1, MODELOS_POR_RAMA

import json, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from joblib import Parallel, delayed
from statsmodels.tsa.holtwinters import ExponentialSmoothing
warnings.filterwarnings('ignore')

pd.set_option('display.width', 230)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

EXPERIMENTO = 'EDA-fix-nivel1-12-segmentacion-volumen'
Y, PISO, H = 'target_demanda_15d', 'pred_media_movil_15d', 15
LLAVES = ['codigo_item', 'sucursal', 'fecha_origen']
FOLDS_EVAL = [2, 3, 4, 5]
FOLDS_BUSQ = list(ev.FOLDS_BUSQUEDA)
VENTANA_VOL, CORTES = 365, (0.5, 0.8)
ESTRATOS = ['cabeza', 'medio', 'cola']
B, SEMILLA = 2000, 0
NUEVAS = ['trail_7', 'trail_28', 'ewma_14', 'ewma_28', 'dias_con_venta_15', 'ratio_trail_nivel',
          'cat_ratio', 'share_cat', 'suc_ratio', 'n_festivos_ventana', 'n_inicio_mes_ventana']
F2 = list(F1) + NUEVAS
HP_CABEZA = dict(n_estimators=200, learning_rate=0.05, num_leaves=8, min_child_samples=20,
                 subsample=0.9, colsample_bytree=0.8, reg_lambda=1.0)     # capacidad chica: la cabeza tiene pocas filas
HP_ESTRATO = {'cabeza': HP_CABEZA, 'medio': MODELOS_POR_RAMA['intermitente']['hparams'],
              'cola': MODELOS_POR_RAMA['intermitente']['hparams']}

verificar_huella('huella_07b.json', ['matriz_as_of_v2.parquet'])
verificar_huella('huella_11.json', ['preds_modelos_relativos_11.parquet'])
verificar_huella('huella_07c.json', ['preds_ablacion_07c.parquet'])
m2 = pd.read_parquet(f'{DW}/matriz_as_of_v2.parquet')
p11 = pd.read_parquet(f'{DW}/preds_modelos_relativos_11.parquet')
p07c = pd.read_parquet(f'{DW}/preds_ablacion_07c.parquet')
print(m2.shape, p11.shape, p07c.shape)

/home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/24 19:39:24 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Huella verificada OK contra huella_07b.json.
Huella verificada OK contra huella_11.json.
Huella verificada OK contra huella_07c.json.
(262429, 38) (236158, 13) (236158, 13)


## 1. Estratos as-of por fold

Volumen (unidades) de los últimos 365 días hábiles hasta `train_end` de cada fold. Un par entra a la cabeza si el volumen acumulado
*antes* de él es menor al 50% del total; al medio si es menor al 80%; el resto es cola.

In [2]:
diario = leer_panel()
fechas, indice_habil = leer_calendario_habil()
ancha = (diario.pivot_table(index='fecha', columns=['codigo_item', 'sucursal'], values='unidades', aggfunc='sum', fill_value=0)
         .reindex(fechas, fill_value=0).astype('float32'))
pares = pd.MultiIndex.from_arrays([ancha.columns.get_level_values(0).astype(str), ancha.columns.get_level_values(1).astype(str)])
ancha_np = ancha.to_numpy()
with open(f'{DW}/huella_07.json') as f:
    FOLDS = json.load(f)['parametros']['folds']

def estratos_de(fold):
    n = fechas.searchsorted(pd.Timestamp(fold['train_end']), side='right')      # días hábiles <= train_end
    vol = ancha_np[max(0, n - VENTANA_VOL):n].sum(axis=0)
    df = pd.DataFrame({'codigo_item': pares.get_level_values(0), 'sucursal': pares.get_level_values(1), 'vol_365d': vol})
    df = df.sort_values('vol_365d', ascending=False, kind='stable').reset_index(drop=True)
    total = df['vol_365d'].sum()
    previo = (df['vol_365d'].cumsum() - df['vol_365d']) / total
    df['estrato'] = np.where(previo < CORTES[0], 'cabeza', np.where(previo < CORTES[1], 'medio', 'cola'))
    df.loc[df['vol_365d'] <= 0, 'estrato'] = 'cola'
    df['fold_id'] = fold['fold_id']
    return df

estr = pd.concat([estratos_de(f) for f in FOLDS if f['fold_id'] in FOLDS_EVAL], ignore_index=True)
print('pares por estrato y fold (definidos as-of, con datos <= train_end):')
print(estr.groupby(['fold_id', 'estrato']).size().unstack()[ESTRATOS])
estr.to_parquet(f'{DW}/estratos_por_fold_12.parquet', index=False)

pares por estrato y fold (definidos as-of, con datos <= train_end):
estrato  cabeza  medio  cola
fold_id                     
2           209   1164  9762
3           203   1180  9752
4           200   1190  9745
5           201   1214  9720


In [3]:
# Unión con las filas de prueba (predicciones de la fase D) y atributos as-of de la matriz v2
cols_m = ['codigo_item', 'sucursal', 'fecha_origen', 'factor_calendario_ventana', 'ewma_28', 'patron_as_of', 'categoria']
rows = p11.merge(m2[cols_m], on=LLAVES, how='left', validate='one_to_one')
for c in ('codigo_item', 'sucursal'):
    rows[c] = rows[c].astype(str)
estr['codigo_item'], estr['sucursal'] = estr['codigo_item'].astype(str), estr['sucursal'].astype(str)
rows = rows.merge(estr[['fold_id', 'codigo_item', 'sucursal', 'estrato', 'vol_365d']], on=['fold_id', 'codigo_item', 'sucursal'],
                  how='left', validate='many_to_one')
assert rows['estrato'].notna().all(), 'hay pares de prueba sin estrato'
rows = rows.rename(columns={'target_demanda_15d': Y}) if Y not in rows.columns else rows
rows['pred_piso'] = rows[PISO]

print('=== Descripción de los estratos en el periodo de PRUEBA (fuera de muestra) ===')
tot = rows.groupby('fold_id')[Y].sum()
d = rows.groupby(['fold_id', 'estrato']).agg(pares=('codigo_item', lambda s: 0), filas=(Y, 'size'), vol=(Y, 'sum'))
d['pares'] = rows.groupby(['fold_id', 'estrato']).apply(lambda g: g[['codigo_item', 'sucursal']].drop_duplicates().shape[0], include_groups=False)
d['%volumen_test'] = d['vol'] / d.index.get_level_values('fold_id').map(tot).to_numpy()
print(d.drop(columns='vol').unstack()['%volumen_test'][ESTRATOS].round(3))
print('\nla cabeza definida as-of sigue concentrando el volumen del periodo de prueba (objetivo ~50%).')
print('\npatrón ADI/CV2 por estrato (% de filas):')
print(pd.crosstab(rows['estrato'], rows['patron_as_of'], normalize='index').reindex(ESTRATOS).round(3))
print('\nrama por estrato (% de filas):')
print(pd.crosstab(rows['estrato'], rows['rama'], normalize='index').reindex(ESTRATOS).round(3))

=== Descripción de los estratos en el periodo de PRUEBA (fuera de muestra) ===
estrato  cabeza  medio  cola
fold_id                     
2         0.454  0.337 0.209
3         0.508  0.300 0.191
4         0.504  0.302 0.194
5         0.496  0.303 0.202

la cabeza definida as-of sigue concentrando el volumen del periodo de prueba (objetivo ~50%).

patrón ADI/CV2 por estrato (% de filas):
patron_as_of  erratico  intermitente  lumpy  suave
estrato                                           
cabeza           0.465         0.000  0.255  0.280
medio            0.032         0.495  0.447  0.026
cola             0.001         0.884  0.115  0.000

rama por estrato (% de filas):
rama     intermitente  suave_no_perecedero  suave_perecedero
estrato                                                     
cabeza          0.720                0.196             0.084
medio           0.974                0.018             0.007
cola            1.000                0.000             0.000


## 2. Candidatos sin entrenar: media móvil y EWMA con factor de calendario

`mm_cal = media_movil · factor_calendario_ventana / factor_medio_de_entrenamiento` (normaliza para que el factor promedio sea 1).
Es un método clásico con estacionalidad de calendario, sin parámetros que ajustar.

In [4]:
rows['pred_mm_cal'] = np.nan
rows['pred_ewma_cal'] = np.nan
for f in FOLDS_EVAL:
    tr = purgar_entrenamiento(m2, f, verbose=False)
    fm = float(tr['factor_calendario_ventana'].mean())
    s = rows['fold_id'] == f
    rows.loc[s, 'pred_mm_cal'] = rows.loc[s, PISO] * rows.loc[s, 'factor_calendario_ventana'] / fm
    rows.loc[s, 'pred_ewma_cal'] = rows.loc[s, 'ewma_28'] * H * rows.loc[s, 'factor_calendario_ventana'] / fm
    print(f'fold {f}: factor medio de entrenamiento = {fm:.3f}')

fold 2: factor medio de entrenamiento = 1.073
fold 3: factor medio de entrenamiento = 1.049
fold 4: factor medio de entrenamiento = 1.079
fold 5: factor medio de entrenamiento = 1.069


## 3. Modelo por serie: Holt-Winters aditivo con estacionalidad semanal (cabeza y medio)

Para cada fila de prueba de la cabeza y el medio se ajusta el modelo a los 365 días hábiles **hasta el origen** (inclusive) de esa serie y se
suma el pronóstico de 15 días. Solo usa historia hasta el origen (as-of).

In [5]:
def ets_suma(x):
    if not np.any(x > 0):
        return 0.0
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            aj = ExponentialSmoothing(x.astype('float64'), trend=None, seasonal='add', seasonal_periods=7,
                                      initialization_method='estimated').fit()
            return float(np.clip(aj.forecast(H), 0, None).sum())
    except Exception:
        return np.nan

t0 = time.time()
sel = rows['estrato'].isin(['cabeza', 'medio'])
sub = rows.loc[sel, ['codigo_item', 'sucursal', 'fecha_origen']]
i_orig = sub['fecha_origen'].map(indice_habil).to_numpy().astype(int)
j_par = pares.get_indexer(pd.MultiIndex.from_arrays([sub['codigo_item'], sub['sucursal']]))
assert (j_par >= 0).all()
series = [ancha_np[max(0, i - 364):i + 1, j] for i, j in zip(i_orig, j_par)]
pred_ets = Parallel(n_jobs=24, batch_size=64)(delayed(ets_suma)(x) for x in series)
rows['pred_ets'] = np.nan
rows.loc[sel, 'pred_ets'] = pred_ets
fallo = rows.loc[sel, 'pred_ets'].isna()
print(f'ETS: {sel.sum():,} ajustes en {time.time()-t0:,.0f}s | fallos: {int(fallo.sum())} ({fallo.mean():.2%}) -> se reemplazan por el piso')
rows.loc[sel & rows['pred_ets'].isna(), 'pred_ets'] = rows.loc[sel & rows['pred_ets'].isna(), PISO]

ETS: 55,093 ajustes en 72s | fallos: 0 (0.00%) -> se reemplazan por el piso


## 4. Modelos relativos entrenados solo con las filas del estrato (`seg_v1`, `seg_v2`)

Target `y / base` con `base = nivel_medio_60d · H + 1`, cuantil 0.5, hiperparámetros fijos: capacidad chica para la cabeza (pocas filas) y los de
`MODELOS_POR_RAMA['intermitente']` para medio y cola. Las filas de entrenamiento son las de folds anteriores con purge cuyo par pertenece al
estrato **según `train_end` del fold evaluado**.

In [6]:
def etiqueta_estrato(df, f):
    e = estr[estr['fold_id'] == f].set_index(['codigo_item', 'sucursal'])['estrato']
    mi = pd.MultiIndex.from_arrays([df['codigo_item'].astype(str), df['sucursal'].astype(str)])
    return e.reindex(mi).to_numpy()

def ajusta_seg(tr, te, cols, hp):
    base_tr, base_te = tr['nivel_medio_60d'] * H + 1.0, te['nivel_medio_60d'] * H + 1.0
    mod = lgb.LGBMRegressor(objective='quantile', alpha=0.5, subsample_freq=1, random_state=42, verbose=-1, **hp)
    mod.fit(tr[cols], tr[Y] / base_tr)
    return np.clip(mod.predict(te[cols]) * base_te.to_numpy(), 0, None)

rows['pred_seg_v1'] = np.nan
rows['pred_seg_v2'] = np.nan
t0 = time.time()
for f in FOLDS_EVAL:
    tr_all = purgar_entrenamiento(m2, f, verbose=False)
    tr_all = tr_all.assign(estrato=etiqueta_estrato(tr_all, f))
    te_all = m2[m2['fold_id'] == f].assign(estrato=lambda d: etiqueta_estrato(d, f))
    for est in ESTRATOS:
        tr, te = tr_all[tr_all['estrato'] == est], te_all[te_all['estrato'] == est]
        if len(tr) < ev.MIN_FILAS_ENTRENAMIENTO or len(te) == 0:
            print(f'fold {f} {est}: train={len(tr)} -- omitido'); continue
        idx = rows.index[(rows['fold_id'] == f) & (rows['estrato'] == est)]
        clave = pd.MultiIndex.from_frame(rows.loc[idx, LLAVES])
        te = te.set_index(LLAVES); te.index = te.index.set_levels(te.index.levels[0].astype(str), level=0).set_levels(te.index.levels[1].astype(str), level=1)
        te = te.reindex(clave)
        for nombre, cols in (('pred_seg_v1', F1), ('pred_seg_v2', F2)):
            rows.loc[idx, nombre] = ajusta_seg(tr, te, cols, HP_ESTRATO[est])
        print(f'fold {f} {est:7s}: train={len(tr):>7,} test={len(idx):>6,} ({time.time()-t0:,.0f}s)')
print('sin predicción seg_v1:', int(rows['pred_seg_v1'].isna().sum()))

fold 2 cabeza : train=    836 test= 2,366 (0s)


fold 2 medio  : train=  4,640 test=14,937 (5s)


fold 2 cola   : train= 15,104 test=51,766 (10s)


fold 3 cabeza : train=  3,446 test= 1,015 (11s)


fold 3 medio  : train= 19,926 test= 5,895 (16s)


...

fold 4 medio  : train= 25,986 test=11,900 (30s)


fold 4 cola   : train= 88,451 test=46,700 (39s)


fold 5 cabeza : train=  6,432 test= 2,412 (40s)


fold 5 medio  : train= 37,959 test=14,568 (47s)


fold 5 cola   : train=134,854 test=60,144 (57s)
sin predicción seg_v1: 0


## 5. Candidatos por estrato: contra el piso y contra el modelo global relativo (`rel`)

`mejora_vs_piso` con IC95. `vs_rel_f2a4` / `vs_rel_f5`: comparación pareada contra `rel` en los folds de búsqueda y en el fold 5
(positivo = el candidato mejora al modelo global relativo). Solo se comparan filas con predicción de ambos.

In [7]:
CANDS = ['pred_piso', 'pred_v1', 'pred_rel', 'pred_rel_w', 'pred_res_mm', 'pred_mezcla50', 'pred_mm_cal', 'pred_ewma_cal',
         'pred_ets', 'pred_seg_v1', 'pred_seg_v2']
REF = 'pred_rel'

def fila_cand(g, est, c):
    s = g.dropna(subset=[c, REF])
    if len(s) == 0:
        return None
    p = ev.mejora_con_ic(s, Y, c, PISO, B, SEMILLA)
    sp = ev.mejora_sin_peor_producto(s, Y, c, PISO, B, SEMILLA)
    fila = dict(estrato=est, cand=c[5:], n_filas=len(s), n_prod=p['n_productos'], wape=p['wape_modelo'], wape_piso=p['wape_piso'],
                mejora_vs_piso=p['mejora_rel'], piso_lo=p['ic_bajo'], piso_hi=p['ic_alto'],
                sin_peor=sp['producto_excluido'], mejora_sin_peor=sp['mejora_rel'])
    if c != REF:
        a = ev.mejora_con_ic(s[s['fold_id'].isin(FOLDS_BUSQ)], Y, c, REF, B, SEMILLA)
        b = ev.mejora_con_ic(s[s['fold_id'] == ev.FOLD_HOLDOUT], Y, c, REF, B, SEMILLA)
        fila.update(vs_rel_f2a4=a['mejora_rel'], vs_rel_lo=a['ic_bajo'], vs_rel_hi=a['ic_alto'], vs_rel_f5=b['mejora_rel'])
    return fila

tabla = pd.DataFrame([f for est in ESTRATOS for c in CANDS if (f := fila_cand(rows[rows['estrato'] == est], est, c)) is not None])
for est in ESTRATOS:
    print(f'\n=== {est.upper()} ===')
    print(tabla[tabla['estrato'] == est].drop(columns=['estrato', 'sin_peor']).to_string(index=False))


=== CABEZA ===
    cand  n_filas  n_prod  wape  wape_piso  mejora_vs_piso  piso_lo  piso_hi  mejora_sin_peor  vs_rel_f2a4  vs_rel_lo  vs_rel_hi  vs_rel_f5
    piso     7793     161 0.250      0.250           0.000    0.000    0.000            0.000        0.008     -0.037      0.043     -0.103
      v1     7793     161 0.270      0.250          -0.082   -0.256    0.034            0.019       -0.143     -0.320     -0.006     -0.011
     rel     7793     161 0.244      0.250           0.022   -0.003    0.051            0.036          NaN        NaN        NaN        NaN
   rel_w     7793     161 0.252      0.250          -0.009   -0.088    0.051            0.037       -0.045     -0.111      0.014      0.003
  res_mm     7793     161 0.245      0.250           0.020    0.002    0.039            0.029        0.013     -0.019      0.036     -0.042
mezcla50     7793     161 0.249      0.250           0.004   -0.063    0.049            0.043       -0.015     -0.069      0.035     -0.028
  mm

## 6. Política híbrida: el mejor candidato por estrato

Se elige, **por estrato y con los folds de búsqueda (2-4)**, el candidato con menor WAPE entre los que cubren todas las filas del estrato
(el piso cuenta como candidato). Luego se evalúa la política completa contra el modelo global relativo, el modelo actual `v1` y el piso, con el
fold 5 como confirmación.

In [8]:
elegido = {}
for est in ESTRATOS:
    g = rows[(rows['estrato'] == est) & rows['fold_id'].isin(FOLDS_BUSQ)]
    w = {c: ev.wape(g[Y], g[c]) for c in CANDS if g[c].notna().all()}
    elegido[est] = min(w, key=w.get)
    print(f'{est:7s}: elegido {elegido[est][5:]:10s} | WAPE folds 2-4: ' + ', '.join(f'{c[5:]} {v:.3f}' for c, v in sorted(w.items(), key=lambda kv: kv[1])[:4]))
rows['pred_hibrido'] = np.select([rows['estrato'] == e for e in ESTRATOS], [rows[elegido[e]] for e in ESTRATOS])

filas = []
for ref in ('pred_piso', 'pred_rel', 'pred_v1'):
    for corte, s in (('todos', rows), ('folds_2a4', rows[rows['fold_id'].isin(FOLDS_BUSQ)]), ('fold_5', rows[rows['fold_id'] == 5])):
        r = ev.mejora_con_ic(s, Y, 'pred_hibrido', ref, B, SEMILLA)
        filas.append(dict(contra=ref[5:], corte=corte, wape_hibrido=r['wape_modelo'], wape_ref=r['wape_piso'],
                          mejora=r['mejora_rel'], lo=r['ic_bajo'], hi=r['ic_alto']))
hib = pd.DataFrame(filas)
print('\nPolítica híbrida (todas las ramas y estratos):'); print(hib.to_string(index=False))
adopta_seg = bool(hib[(hib.contra == 'rel') & (hib.corte == 'folds_2a4')]['lo'].iloc[0] > 0
                  and hib[(hib.contra == 'rel') & (hib.corte == 'fold_5')]['mejora'].iloc[0] > 0)
print('\n¿La segmentación mejora al modelo global relativo (IC95 > 0 en 2-4 y fold 5 no la contradice)?', 'SÍ' if adopta_seg else 'NO')

cabeza : elegido mm_cal     | WAPE folds 2-4: mm_cal 0.248, seg_v2 0.251, res_mm 0.254, piso 0.255
medio  : elegido seg_v2     | WAPE folds 2-4: seg_v2 0.385, seg_v1 0.386, v1 0.388, mezcla50 0.389
cola   : elegido seg_v2     | WAPE folds 2-4: seg_v2 0.615, seg_v1 0.616, v1 0.619, rel 0.620



Política híbrida (todas las ramas y estratos):
contra     corte  wape_hibrido  wape_ref  mejora     lo     hi
  piso     todos         0.361     0.380   0.050  0.043  0.055
  piso folds_2a4         0.365     0.384   0.051  0.043  0.058
  piso    fold_5         0.354     0.371   0.046  0.040  0.053
   rel     todos         0.361     0.364   0.007 -0.001  0.017
   rel folds_2a4         0.365     0.371   0.018  0.005  0.036
   rel    fold_5         0.354     0.346  -0.021 -0.032 -0.010
    v1     todos         0.361     0.376   0.040  0.002  0.098
    v1 folds_2a4         0.365     0.389   0.062  0.008  0.143
    v1    fold_5         0.354     0.348  -0.016 -0.026 -0.006

¿La segmentación mejora al modelo global relativo (IC95 > 0 en 2-4 y fold 5 no la contradice)? NO


## 7. ¿Cambia el valor de las features por estrato? (feature engineering)

Con las predicciones de la ablación 07c (modelos globales con distintos grupos de features, ya con purge), se mide por estrato cuánto mejora
cada grupo sobre `v1`. Si el efecto difiere entre cabeza y cola, las features conviene diseñarlas por estrato.

In [9]:
for c in ('codigo_item', 'sucursal'):
    p07c[c] = p07c[c].astype(str)
fe = p07c.merge(estr[['fold_id', 'codigo_item', 'sucursal', 'estrato']], on=['fold_id', 'codigo_item', 'sucursal'], how='left')
assert fe['estrato'].notna().all()
cfgs = [c for c in fe.columns if c.startswith('pred_v1+') or c == 'pred_v2_completa']   # solo las configuraciones de features (no la media móvil)
filas = []
for est in ESTRATOS:
    g = fe[fe['estrato'] == est]
    for c in cfgs:
        a = ev.mejora_con_ic(g, Y, c, 'pred_v1', B, SEMILLA)
        filas.append(dict(estrato=est, config=c[5:], n_prod=a['n_productos'], mejora_vs_v1=a['mejora_rel'], lo=a['ic_bajo'], hi=a['ic_alto'],
                          distinguible=bool(a['ic_bajo'] > 0 or a['ic_alto'] < 0)))
fe_tab = pd.DataFrame(filas)
print(fe_tab.pivot(index='config', columns='estrato', values='mejora_vs_v1')[ESTRATOS].round(4))
print('\nIC95 distinguible de cero (True = el grupo cambia el resultado en ese estrato):')
print(fe_tab.pivot(index='config', columns='estrato', values='distinguible')[ESTRATOS])
print('\nAdemás, seg_v2 (25 features) frente a seg_v1 (14) dentro de cada estrato:')
for est in ESTRATOS:
    g = rows[rows['estrato'] == est].dropna(subset=['pred_seg_v1', 'pred_seg_v2'])
    if len(g):
        a = ev.mejora_con_ic(g, Y, 'pred_seg_v2', 'pred_seg_v1', B, SEMILLA)
        print(f'  {est:7s}: {a["mejora_rel"]:+.2%} IC [{a["ic_bajo"]:+.2%}, {a["ic_alto"]:+.2%}]')

estrato        cabeza  medio  cola
config                            
v1+calendario   0.001 -0.004 0.000
v1+contexto    -0.012  0.002 0.001
v1+memoria     -0.004 -0.006 0.001
v1+momentum     0.008  0.002 0.001
v2_completa    -0.026 -0.005 0.002

IC95 distinguible de cero (True = el grupo cambia el resultado en ese estrato):
estrato        cabeza  medio   cola
config                             
v1+calendario   False   True  False
v1+contexto      True   True   True
v1+memoria      False   True  False
v1+momentum      True  False   True
v2_completa      True   True   True

Además, seg_v2 (25 features) frente a seg_v1 (14) dentro de cada estrato:
  cabeza : +3.21% IC [+1.07%, +4.92%]
  medio  : +0.33% IC [+0.01%, +0.64%]


  cola   : +0.11% IC [-0.00%, +0.23%]


## 8. Decisión mecánica, artefactos y registro en MLflow

In [10]:
dec = {'notebook': '12_segmentacion_volumen', 'fecha_generacion': pd.Timestamp.now().isoformat(),
       'definicion_estratos': {'ventana_dias_habiles': VENTANA_VOL, 'cortes_volumen_acumulado': list(CORTES), 'as_of': 'train_end de cada fold'},
       'politica_holdout': ev.POLITICA_HOLDOUT, 'hiperparametros': 'fijos (sin busqueda)', 'referencia': REF,
       'pares_por_estrato': {str(f): estr[estr.fold_id == f].groupby('estrato').size().to_dict() for f in FOLDS_EVAL},
       'estratos': {}, 'hibrido': {'elegido_por_estrato': {e: elegido[e][5:] for e in ESTRATOS},
                                   'mejora_vs_rel_f2a4': float(hib[(hib.contra == 'rel') & (hib.corte == 'folds_2a4')]['mejora'].iloc[0]),
                                   'ic_vs_rel_f2a4': [float(hib[(hib.contra == 'rel') & (hib.corte == 'folds_2a4')]['lo'].iloc[0]),
                                                      float(hib[(hib.contra == 'rel') & (hib.corte == 'folds_2a4')]['hi'].iloc[0])],
                                   'mejora_vs_rel_f5': float(hib[(hib.contra == 'rel') & (hib.corte == 'fold_5')]['mejora'].iloc[0]),
                                   'mejora_vs_piso': float(hib[(hib.contra == 'piso') & (hib.corte == 'todos')]['mejora'].iloc[0]),
                                   'ic_vs_piso': [float(hib[(hib.contra == 'piso') & (hib.corte == 'todos')]['lo'].iloc[0]),
                                                  float(hib[(hib.contra == 'piso') & (hib.corte == 'todos')]['hi'].iloc[0])],
                                   'segmentacion_mejora_al_global_relativo': adopta_seg}}
for est in ESTRATOS:
    t = tabla[(tabla.estrato == est) & (tabla.cand != 'rel')].dropna(subset=['vs_rel_f2a4'])
    mejor = t.loc[t['vs_rel_f2a4'].idxmax()]
    dec['estratos'][est] = dict(mejor_vs_rel=mejor['cand'], vs_rel_f2a4=float(mejor['vs_rel_f2a4']), ic=[float(mejor['vs_rel_lo']), float(mejor['vs_rel_hi'])],
                                vs_rel_f5=float(mejor['vs_rel_f5']), mejora_vs_piso=float(mejor['mejora_vs_piso']),
                                adopta_sobre_rel=bool(mejor['vs_rel_lo'] > 0 and mejor['vs_rel_f5'] > 0))
    print(f"{est:7s}: mejor candidato contra `rel`: {mejor['cand']:10s} {mejor['vs_rel_f2a4']:+.2%} IC [{mejor['vs_rel_lo']:+.2%}, {mejor['vs_rel_hi']:+.2%}] "
          f"fold 5 {mejor['vs_rel_f5']:+.2%} -> " + ('ADOPTA' if dec['estratos'][est]['adopta_sobre_rel'] else 'no adopta'))
with open(f'{DW}/decision_12.json', 'w') as f:
    json.dump(dec, f, indent=2, ensure_ascii=False)
tabla.to_csv(f'{DW}/resultados_segmentacion_12.csv', index=False)
fe_tab.to_csv(f'{DW}/fe_por_estrato_12.csv', index=False)
hib.to_csv(f'{DW}/hibrido_12.csv', index=False)
rows.to_parquet(f'{DW}/preds_segmentacion_12.parquet', index=False)
huella = {'notebook': '12_segmentacion_volumen', 'fecha_generacion': pd.Timestamp.now().isoformat(),
          'entradas': registrar_huella(['matriz_as_of_v2.parquet', 'preds_modelos_relativos_11.parquet', 'preds_ablacion_07c.parquet', 'panel_diario.parquet']),
          'salidas': registrar_huella(['decision_12.json', 'resultados_segmentacion_12.csv', 'preds_segmentacion_12.parquet', 'estratos_por_fold_12.parquet'])}
with open(f'{DW}/huella_12.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
for _, r in tabla.iterrows():
    metrics = {k: r[k] for k in ['n_filas', 'n_prod', 'wape', 'wape_piso', 'mejora_vs_piso', 'piso_lo', 'piso_hi', 'mejora_sin_peor'] if pd.notna(r[k])}
    metrics.update({k: r[k] for k in ['vs_rel_f2a4', 'vs_rel_lo', 'vs_rel_hi', 'vs_rel_f5'] if k in r and pd.notna(r[k])})
    ev.registrar_mlflow(EXPERIMENTO, f"{r['estrato']}__{r['cand']}",
        tags={'fase': 'E', 'notebook': '12', 'estrato': r['estrato'], 'cand': r['cand'], 'rama_git': 'fix/eda-modelado-nivel1',
              'elegido_hibrido': str(elegido[r['estrato']][5:] == r['cand'])},
        params={'ventana_vol': VENTANA_VOL, 'cortes': str(CORTES), 'referencia': REF, 'bootstrap_B': B, 'hiperparametros': 'fijos'},
        metrics=metrics, artefactos=[f'{DW}/decision_12.json'] if (r['estrato'] == 'cola' and r['cand'] == 'rel') else ())
print(f'{len(tabla)} runs registrados en {EXPERIMENTO}')

cabeza : mejor candidato contra `rel`: mm_cal     +3.61% IC [-0.34%, +7.22%] fold 5 -6.94% -> no adopta
medio  : mejor candidato contra `rel`: seg_v2     +1.11% IC [+0.48%, +1.76%] fold 5 +0.60% -> ADOPTA
cola   : mejor candidato contra `rel`: seg_v2     +0.72% IC [+0.46%, +0.97%] fold 5 -0.34% -> no adopta


🏃 View run cabeza__piso at: http://127.0.0.1:5000/#/experiments/19/runs/c1003306cd9148ba99083134f474ecff
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/19


🏃 View run cabeza__v1 at: http://127.0.0.1:5000/#/experiments/19/runs/66c59b02a177400c82c6ba072e67653f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/19


🏃 View run cabeza__rel at: http://127.0.0.1:5000/#/experiments/19/runs/447f4af2427b4550850552bbca730a4d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/19


🏃 View run cabeza__rel_w at: http://127.0.0.1:5000/#/experiments/19/runs/123f94c34ff04d23abf2e4a1496e15cb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/19


...

🏃 View run cola__mezcla50 at: http://127.0.0.1:5000/#/experiments/19/runs/9a0a2bce1e0f42c9a4642005a95ecdd1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/19


🏃 View run cola__mm_cal at: http://127.0.0.1:5000/#/experiments/19/runs/53b43be9df5f4f3cba94f8280f4cb8ce
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/19


🏃 View run cola__ewma_cal at: http://127.0.0.1:5000/#/experiments/19/runs/daf5d9a6c19648d3ba1ff9facaa18b51
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/19


🏃 View run cola__seg_v1 at: http://127.0.0.1:5000/#/experiments/19/runs/470c1f37b48244989f26dcebc9c817d9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/19


🏃 View run cola__seg_v2 at: http://127.0.0.1:5000/#/experiments/19/runs/e3525f3383a0455dbdc9f2883eb88e52
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/19
32 runs registrados en EDA-fix-nivel1-12-segmentacion-volumen
